# 02 - Prompts and chains (LCEL)

Notebook 1 was about a single primitive: the model. Real applications never call the model with a hardcoded string. They build messages from user input and from variables, and they often want the model's output reshaped before handing it back to the caller.

That is what this notebook is about:

- **Prompt templates** turn a parameterized template into a list of messages
- **Output parsers** turn an `AIMessage` into something more useful (a string, a dict, a Pydantic object)
- **The pipe operator** (`|`) connects them all into a *Runnable*, which is just a fancy word for a callable that supports `invoke`, `stream`, and `batch`

If you remember nothing else, remember this pattern: `prompt | model | parser`. That is LCEL. That is the spine of LangChain.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-4o-mini")

## A simple prompt template

The simplest case: a single user-style message with one variable.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Tell me a one-line joke about {topic}.")

messages = prompt.invoke({"topic": "databases"})
messages

Notice that `prompt.invoke(...)` does **not** call any model. All it does is fill in the `{topic}` placeholder and return a `ChatPromptValue` object that wraps a list of messages. The model call is a separate step.

## Multi-message templates: `from_messages`

More common in practice: a system message that sets behavior plus a human message that holds the actual user input.

In [ ]:
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You translate {input_language} to {output_language}. Return only the translation, no commentary."),
    ("human", "{text}"),
])

translate_prompt.invoke({
    "input_language": "English",
    "output_language": "French",
    "text": "I love programming.",
})

Each tuple is `(role, template)`. Roles are `system`, `human`, `ai`, `tool`. The template part can have any number of `{variable}` placeholders, and they all get filled from the dict you pass to `invoke`.

## The pipe operator: building a chain

Now the move. Both `prompt` and `model` are *Runnables*. Every Runnable supports the same trio: `invoke`, `stream`, `batch`. And Runnables compose with `|`. So:

```python
chain = prompt | model
```

is itself a Runnable. Calling `chain.invoke({...})` is the same as calling `model.invoke(prompt.invoke({...}))`, just written without the visual nesting.

In [ ]:
chain = translate_prompt | model

result = chain.invoke({
    "input_language": "English",
    "output_language": "French",
    "text": "I love programming.",
})

print(type(result).__name__, "|", result.text)

### Why is `|` a thing?

Three reasons it is worth the syntactic trick:

1. **Uniform interface**. Every step in the chain accepts the previous step's output. You do not have to manually thread variables.
2. **Free streaming and batching**. Because both `prompt` and `model` are Runnables, the chain *itself* gets `chain.stream(...)` and `chain.batch(...)` for free.
3. **Cheap to refactor**. Want to add a step? Insert it in the pipe. Want to remove the model and just inspect what the prompt produces? Drop the right side.

Once you internalize this, half of LangChain stops looking like a framework and starts looking like a pipe.

## Output parsers: getting plain values back

By default a chain ends with an `AIMessage`. Calling `.text` every time is fine but verbose. Parsers do that flatten step for you.

The simplest one is `StrOutputParser`. It just returns `message.text`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = translate_prompt | model | StrOutputParser()

translation = chain.invoke({
    "input_language": "English",
    "output_language": "German",
    "text": "Where is the train station?",
})

print(repr(translation))

Now `chain.invoke({...})` returns a `str`. That tiny change makes a chain feel like a regular Python function, which makes it much easier to plug into the rest of your code.

## Parsing JSON output

If you ask the model for JSON, you can have the parser convert the text into a Python dict for you.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reply with a JSON object that has keys 'title' and 'year'. No prose."),
    ("human", "Pick a famous movie about {topic}."),
])

json_chain = json_prompt | model | JsonOutputParser()

out = json_chain.invoke({"topic": "dreams"})
print(type(out).__name__, out)
print("title alone:", out["title"])

**Heads up**: `JsonOutputParser` will *try* to fix slightly malformed JSON, but it is not bulletproof. For anything you actually care about, use `with_structured_output` (covered in notebook 3). It uses the provider's native structured output mode and is far more reliable than asking nicely in the system prompt.

## Pydantic output parser

Same idea, but you define a Pydantic schema and get back a typed object. The parser also injects formatting instructions into your prompt automatically.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Movie(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Year of release")
    director: str = Field(description="Director's full name")

parser = PydanticOutputParser(pydantic_object=Movie)

movie_prompt = ChatPromptTemplate.from_messages([
    ("system", "Pick a famous movie. {format_instructions}"),
    ("human", "Topic: {topic}"),
]).partial(format_instructions=parser.get_format_instructions())

movie_chain = movie_prompt | model | parser

movie = movie_chain.invoke({"topic": "time travel"})
print(type(movie).__name__)
print(movie)
print("director:", movie.director)

Two things to notice:

1. `parser.get_format_instructions()` returns a string that explains the JSON schema to the model. We bake it into the system prompt with `.partial(...)` so it is fixed across every call.
2. The chain returns a real `Movie` instance with attribute access (`movie.director`), not a dict.

**Insight on `.partial`**: prompt templates are *partial-friendly*. You can fix some variables now and leave the rest for later. Useful for any value that does not change between calls (model name, format spec, today's date).

## `MessagesPlaceholder`: chat history as a variable

Real chat needs history. You do not want to bake the history into the template as a fixed list of tuples, because the history grows. The trick is `MessagesPlaceholder`, which leaves a slot in the template that gets filled with a *list of messages* at runtime.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep replies under 30 words."),
    MessagesPlaceholder("history"),
    ("human", "{user_input}"),
])

chat_chain = chat_prompt | model | StrOutputParser()

history = [
    HumanMessage("My name is Sam."),
    AIMessage("Nice to meet you, Sam."),
]

reply = chat_chain.invoke({
    "history": history,
    "user_input": "What is my name?",
})

print(reply)

**Insight**: this is how chat apps maintain memory in a stateless model world. You keep `history` somewhere (in-memory list, Redis, a database) and re-inject it on every turn. LangChain has higher-level helpers (`RunnableWithMessageHistory`, the `langgraph` checkpointer) for this, but they all sit on top of the same idea: pass the past messages as a list, every turn.

## Streaming and batching a chain

Because a chain is a Runnable, it inherits `stream` and `batch` for free. No new API to learn.

In [ ]:
chain = translate_prompt | model | StrOutputParser()

for chunk in chain.stream({
    "input_language": "English",
    "output_language": "Spanish",
    "text": "The library closes in ten minutes. Please bring back any books you borrowed.",
}):
    print(chunk, end="", flush=True)
print()

In [ ]:
translations = chain.batch([
    {"input_language": "English", "output_language": "French",  "text": "Good morning."},
    {"input_language": "English", "output_language": "German",  "text": "Good morning."},
    {"input_language": "English", "output_language": "Spanish", "text": "Good morning."},
])

for t in translations:
    print(t)

Notice the chain `stream` yields *strings*, not `AIMessageChunk` objects. The `StrOutputParser` at the end of the pipe knows how to handle streaming and unwraps each chunk's text for you. Different parsers handle streaming differently (e.g. `JsonOutputParser` yields progressively-completing dicts), which is genuinely useful but a deeper topic for later.

## Recap

What you can now do:

- write a `ChatPromptTemplate` with `{variable}` placeholders
- compose `prompt | model | parser` into a callable chain
- swap parsers to control the chain's output type (string, dict, Pydantic object)
- inject conversation history with `MessagesPlaceholder`
- stream and batch a chain just like a model

What is still missing:

- the model deciding to call a function on its own (tool calling)
- a more robust way to get structured output than asking via the prompt

Both of those are notebook 3.